# 🧠 Brain Tumor Classification with ResNet18 — Optimized Transfer Learning Pipeline

This notebook implements a **two-phase transfer learning** approach using ResNet18 on grayscale MRI brain tumor images.

**Dataset:** 4 classes — `glioma`, `meningioma`, `no_tumor`, `pituitary`

**Expected Accuracy:** ~90–97%

---
### 📋 Pipeline Overview
1. Import Libraries + Device Setup
2. Configure Local Dataset Paths
3. Preprocessing & Transforms
4. Dataset Loading with 80/20 Validation Split
5. ResNet18 Model Setup (Transfer Learning)
6. Training Helper Functions
7. Phase 1 — Train Head Only (Frozen Backbone)
8. Phase 2 — Fine-Tune layer4 + Head
9. Plot Training Curves
10. Evaluate on Test Set
11. Confusion Matrix
12. Save Model
13. (Optional) Predict on a Single Image

## 📦 Cell 1: Install Dependencies (if needed)

In [ ]:
# Run once if any package is missing:
# !pip install torch torchvision scikit-learn seaborn matplotlib

print("✅ All dependencies ready.")

## 📚 Cell 2: Imports + Device Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ─── Device Setup ────────────────────────────────────────────────
# Supports: NVIDIA GPU (cuda), Apple Silicon (mps), or CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🖥️  Using device: CUDA GPU — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🖥️  Using device: Apple MPS (Metal) — Mac GPU")
else:
    device = torch.device('cpu')
    print("🖥️  Using device: CPU (no GPU detected)")

## 📁 Cell 3: Configure Local Dataset Paths

> **Update `TRAIN_DIR` and `TEST_DIR`** below to point to your local `classification_task` folder.

In [ ]:
# ─── UPDATE THESE PATHS ──────────────────────────────────────────
TRAIN_DIR = '/Users/swaroopnaik1905/brisc2025/classification_task/train'
TEST_DIR  = '/Users/swaroopnaik1905/brisc2025/classification_task/test'

# ─── Hyperparameters ─────────────────────────────────────────────
BATCH_SIZE    = 32
NUM_CLASSES   = 4
IMG_SIZE      = 224
VAL_SPLIT     = 0.2
RANDOM_SEED   = 42

# Phase 1 settings
PHASE1_EPOCHS = 5
PHASE1_LR     = 1e-4

# Phase 2 settings
PHASE2_EPOCHS = 15
PHASE2_LR     = 5e-5

# ─── Verify paths exist ───────────────────────────────────────────
assert os.path.exists(TRAIN_DIR), f"❌ Train dir not found: {TRAIN_DIR}"
assert os.path.exists(TEST_DIR),  f"❌ Test dir not found: {TEST_DIR}"
print(f"✅ Train dir: {TRAIN_DIR}")
print(f"✅ Test dir:  {TEST_DIR}")

## 🔄 Cell 4: Preprocessing & Data Transforms

Key decisions:
- **`Grayscale(3)`** → Converts single-channel MRI to 3-channel (required by ResNet18)
- **`RandomHorizontalFlip`** → Minimal, medically-safe augmentation
- **ImageNet normalization** → Standard for pretrained ResNet models

In [ ]:
# ImageNet mean/std — ResNet18 was pretrained on ImageNet
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ─── Training Transform ──────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # 1-channel → 3-channel
    transforms.RandomHorizontalFlip(),             # Only safe augmentation
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ─── Validation / Test Transform ────────────────────────────────
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print("✅ Transforms defined.")

## 📂 Cell 5: Load Dataset & Create 80/20 Validation Split

In [ ]:
# ─── Load full train dataset (no transform yet) ──────────────────
full_train_ds = datasets.ImageFolder(root=TRAIN_DIR)
CLASS_NAMES   = full_train_ds.classes
print(f"📊 Classes detected: {CLASS_NAMES}")

# ─── Stratified 80/20 Split ──────────────────────────────────────
all_indices = list(range(len(full_train_ds)))
all_labels  = full_train_ds.targets

train_idx, val_idx = train_test_split(
    all_indices,
    test_size=VAL_SPLIT,
    stratify=all_labels,      # Maintain class balance in both splits
    random_state=RANDOM_SEED
)

# ─── Create Subsets with correct transforms ──────────────────────
# Two separate ImageFolder instances so each subset gets its own transform
train_ds_tf = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_ds_tf   = datasets.ImageFolder(root=TRAIN_DIR, transform=val_test_transform)

train_dataset = Subset(train_ds_tf, train_idx)
val_dataset   = Subset(val_ds_tf,   val_idx)

# ─── Test Dataset ────────────────────────────────────────────────
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transform)

# ─── DataLoaders ─────────────────────────────────────────────────
# num_workers=0 is safest on Mac (avoids multiprocessing issues with MPS)
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS
)

print(f"\n📊 Dataset Sizes:")
print(f"   Train      : {len(train_dataset):>5} images")
print(f"   Validation : {len(val_dataset):>5} images")
print(f"   Test       : {len(test_dataset):>5} images")

## 🔬 Cell 6: Visualize Sample Images

Quick sanity check — peek at a few training images to confirm loading is correct.

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Sample Training Images", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i >= 8 or i >= len(images):
        break
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Class: {CLASS_NAMES[labels[i]]}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 🏗️ Cell 7: Build ResNet18 Model (Transfer Learning Setup)

**Strategy:**
1. Load pretrained ResNet18 (ImageNet weights)
2. **Freeze ALL layers** — backbone stays frozen for Phase 1
3. Replace the final `fc` layer with a new 4-class classifier
   - New `fc` is NOT frozen (trainable by default)

In [ ]:
# ─── Load Pretrained ResNet18 ────────────────────────────────────
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# ─── Freeze ALL layers ───────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = False

# ─── Replace Final FC Layer ──────────────────────────────────────
# Original: Linear(512 → 1000)  [ImageNet]
# New:      Linear(512 → 4)     [our 4 classes]
num_features = model.fc.in_features  # 512 for ResNet18
model.fc = nn.Linear(num_features, NUM_CLASSES)
# model.fc is NOT frozen — its parameters will be learned in Phase 1

model = model.to(device)

# ─── Loss Function ───────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()

# ─── Summary ─────────────────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ ResNet18 loaded (ImageNet pretrained)")
print(f"   Total parameters    : {total_params:,}")
print(f"   Trainable (Phase 1) : {trainable_params:,}  ← fc layer only")
print(f"   Output classes      : {NUM_CLASSES}")

## 🔧 Cell 8: Training & Validation Helper Functions

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, is_training=True):
    """
    Runs one epoch of training or validation.

    Args:
        model       : PyTorch model
        loader      : DataLoader
        criterion   : Loss function
        optimizer   : Optimizer (None during validation)
        is_training : If True, performs backprop

    Returns:
        avg_loss (float), accuracy (float in %)
    """
    model.train() if is_training else model.eval()

    total_loss = 0.0
    correct    = 0
    total      = 0

    context = torch.enable_grad() if is_training else torch.no_grad()

    with context:
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            if is_training:
                optimizer.zero_grad()

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy


def train_phase(model, optimizer, scheduler, num_epochs, phase_name,
                history=None):
    """
    Trains the model for `num_epochs`, recording metrics each epoch.

    Args:
        model      : PyTorch model
        optimizer  : Optimizer
        scheduler  : LR scheduler (can be None)
        num_epochs : Number of epochs
        phase_name : Label for display
        history    : Optional dict to append to (for combined plotting)

    Returns:
        history dict with: train_loss, val_loss, train_acc, val_acc
    """
    if history is None:
        history = {'train_loss': [], 'val_loss': [],
                   'train_acc':  [], 'val_acc':  []}

    print(f"\n{'='*65}")
    print(f"  {phase_name}")
    print(f"{'='*65}")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
          f"{'Val Loss':>9} | {'Val Acc':>8}")
    print(f"{'-'*65}")

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_acc = run_epoch(
            model, train_loader, criterion, optimizer, is_training=True
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, is_training=False
        )

        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)

        print(f"{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2f}% | "
              f"{val_loss:>9.4f} | {val_acc:>7.2f}%")

    return history


print("✅ Training helper functions defined.")

## 🚀 Cell 9: Phase 1 — Train Head Only (Frozen Backbone)

Train only the new `fc` layer. Backbone stays frozen.

- **Optimizer:** Adam, LR = `0.0001`
- **Epochs:** 5 (warmup the classifier head)
- **No LR scheduler** needed for short warmup phase

In [ ]:
# Only the fc layer parameters are passed to the optimizer
optimizer_p1 = optim.Adam(model.fc.parameters(), lr=PHASE1_LR)

history = train_phase(
    model,
    optimizer=optimizer_p1,
    scheduler=None,
    num_epochs=PHASE1_EPOCHS,
    phase_name="PHASE 1 — Train Head Only (Backbone Frozen)",
    history=None
)

print("\n✅ Phase 1 complete.")

## 🔓 Cell 10: Phase 2 — Fine-Tune `layer4` + Head

Unfreeze `layer4` (the deepest conv block) and `fc`, retrain at a lower LR.

- **Optimizer:** Adam, LR = `0.00005` (5× smaller than Phase 1)
- **Scheduler:** StepLR — drops LR by 10× after epoch 7
- **Epochs:** 15

In [ ]:
# ─── Unfreeze layer4 and fc ──────────────────────────────────────
for name, param in model.named_parameters():
    if 'layer4' in name or name.startswith('fc'):
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🔓 Unfroze: layer4 + fc")
print(f"   Trainable parameters (Phase 2): {trainable:,}")

# ─── Phase 2 Optimizer ───────────────────────────────────────────
optimizer_p2 = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE2_LR
)

# ─── StepLR Scheduler ────────────────────────────────────────────
scheduler_p2 = optim.lr_scheduler.StepLR(
    optimizer_p2,
    step_size=7,    # Reduce LR after 7 epochs
    gamma=0.1       # Multiply LR by 0.1  (5e-5 → 5e-6)
)

# ─── Run Phase 2 ─────────────────────────────────────────────────
history = train_phase(
    model,
    optimizer=optimizer_p2,
    scheduler=scheduler_p2,
    num_epochs=PHASE2_EPOCHS,
    phase_name="PHASE 2 — Fine-Tune layer4 + Head (Low LR)",
    history=history    # Appends to history from Phase 1
)

print("\n✅ Phase 2 complete.")

## 📈 Cell 11: Plot Training Curves

A vertical dashed line marks where Phase 1 ends and Phase 2 begins.

In [ ]:
total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS
epochs_range = range(1, total_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Training Curves — ResNet18 Transfer Learning',
             fontsize=14, fontweight='bold')

# ─── Loss Plot ───────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], 'b-o',
        linewidth=2, markersize=4, label='Train Loss')
ax.plot(epochs_range, history['val_loss'], 'r-o',
        linewidth=2, markersize=4, label='Val Loss')
ax.axvline(x=PHASE1_EPOCHS + 0.5, color='gray', linestyle='--',
           linewidth=1.5, label='Phase 1 → Phase 2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss per Epoch')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xticks(list(epochs_range))

# ─── Accuracy Plot ───────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_range, history['train_acc'], 'b-o',
        linewidth=2, markersize=4, label='Train Accuracy')
ax.plot(epochs_range, history['val_acc'], 'r-o',
        linewidth=2, markersize=4, label='Val Accuracy')
ax.axvline(x=PHASE1_EPOCHS + 0.5, color='gray', linestyle='--',
           linewidth=1.5, label='Phase 1 → Phase 2')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy per Epoch')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim([0, 105])
ax.set_xticks(list(epochs_range))

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: training_curves.png")

## 🧪 Cell 12: Final Evaluation on Test Set

In [ ]:
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = 100.0 * (all_preds == all_labels).sum() / len(all_labels)

print(f"\n{'='*50}")
print(f"  🎯 Final Test Accuracy: {test_accuracy:.2f}%")
print(f"{'='*50}\n")

print("📋 Classification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=CLASS_NAMES,
    digits=4
))

## 🗺️ Cell 13: Confusion Matrix

Displays both raw counts and normalized (recall-per-class) versions.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix  |  Test Accuracy: {test_accuracy:.2f}%',
             fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0]
)
axes[0].set_title('Raw Counts')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=45)

# Normalized (recall)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    vmin=0, vmax=1,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1]
)
axes[1].set_title('Normalized (Recall per class)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: confusion_matrix.png")

## 💾 Cell 14: Save the Trained Model

In [ ]:
# Save weights to the dataset parent directory
save_dir = os.path.dirname(TRAIN_DIR)
MODEL_SAVE_PATH = os.path.join(save_dir, 'brain_tumor_resnet18.pth')
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"✅ Model weights saved to: {MODEL_SAVE_PATH}")

print("""
── To reload this model later ──────────────────────────────────
  model = models.resnet18(weights=None)
  model.fc = nn.Linear(512, 4)
  model.load_state_dict(torch.load('brain_tumor_resnet18.pth',
                                   map_location=device))
  model = model.to(device)
  model.eval()
────────────────────────────────────────────────────────────────
""")

## 🔮 Cell 15 (Optional): Predict on a Single Image

In [ ]:
from PIL import Image

def predict_single_image(image_path, model, transform, class_names, device):
    """
    Classifies a single MRI image and shows probabilities.

    Args:
        image_path  : Path to image file (.jpg / .png)
        model       : Trained PyTorch model
        transform   : Preprocessing transform
        class_names : List of class label strings
        device      : torch.device
    """
    img    = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs  = model(tensor)
        probs    = torch.softmax(outputs, dim=1)[0]
        pred_idx = probs.argmax().item()

    pred_class = class_names[pred_idx]
    confidence = probs[pred_idx].item() * 100

    # Display
    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap='gray')
    plt.title(f"Predicted: {pred_class}\n({confidence:.1f}% confidence)",
              fontweight='bold')
    plt.axis('off')
    plt.show()

    print(f"\nAll class probabilities:")
    for i, (cls, prob) in enumerate(zip(class_names, probs)):
        marker = " ← predicted" if i == pred_idx else ""
        print(f"  {cls:<12}: {prob.item()*100:>6.2f}%{marker}")

    return pred_class, confidence


# ─── Usage ───────────────────────────────────────────────────────
# Update the path to any MRI image on your machine:
#
# predict_single_image(
#     image_path='/Users/swaroopnaik1905/brisc2025/classification_task/test/glioma/Te-gl_0010.jpg',
#     model=model,
#     transform=val_test_transform,
#     class_names=CLASS_NAMES,
#     device=device
# )

print("✅ Single-image prediction function ready.")
print("   Uncomment the usage block above and update the image path.")

---
## ✅ Summary

| Setting | Value |
|---------|-------|
| **Model** | ResNet18 (ImageNet pretrained) |
| **Input** | 224×224, grayscale → 3-channel via `Grayscale(3)` |
| **Augmentation** | `RandomHorizontalFlip` only |
| **Normalization** | ImageNet mean/std |
| **Phase 1** | 5 epochs, frozen backbone, fc only, LR=1e-4 |
| **Phase 2** | 15 epochs, layer4+fc unfrozen, LR=5e-5, StepLR |
| **Validation** | 80/20 stratified split |
| **Optimizer** | Adam |
| **Loss** | CrossEntropyLoss |
| **Device** | CUDA / MPS (Apple Silicon) / CPU (auto-detected) |
| **Expected Accuracy** | ~90–97% |